In [1]:
from dotenv import load_dotenv
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

load_dotenv()

# Define the model to use throughout this notebook
# Using Opus 4.6 for its superior planning and reasoning capabilities
MODEL = "claude-opus-4-6"
print(f"📋 Notebook configured to use: {MODEL}")

📋 Notebook configured to use: claude-opus-4-6


# 01 - 幕僚长智能体

#### 引言

在笔记本 00 中，我们构建了一个简单的检索智能体。在本笔记本中，我们将逐步介绍用于构建综合型智能体的 Claude Code SDK 关键特性。对每个介绍的特性，我们会说明：
- **是什么**：该特性是什么
- **为什么**：该特性能做什么，以及为什么要使用它
- **怎么做**：一个展示如何使用它的最小实现

如果你熟悉 Claude Code，你会发现 SDK 提供了特性对等，让你能以编程化的无界面方式利用 Claude Code 的全部能力。

#### 场景

在本笔记本中，我们将为一家刚完成 $10M A 轮融资的 50 人初创公司构建一个 **AI 幕僚长**。CEO 需要数据驱动的洞察，在激进增长与财务可持续性之间取得平衡。

最终的幕僚长智能体会：
- **为不同领域协调专业化的子智能体**
- **汇总来自多个来源的洞察**
- **给出带有可执行建议的高管摘要**

## 基础特性

### 特性 0：用 [CLAUDE.md](https://www.anthropic.com/engineering/claude-code-best-practices) 实现记忆

**是什么**：`CLAUDE.md` 文件充当智能体的持久记忆和指令。当它位于项目目录中时，Claude Code 会在初始化智能体时自动读取并纳入这份上下文。

**为什么**：不必在每次交互中重复提供项目上下文、团队偏好或规范，只需在 `CLAUDE.md` 中定义一次。这能保证行为一致，并通过避免重复说明减少词元用量。

**怎么做**：
- 在工作目录中放置 `CLAUDE.md` 文件——在我们的示例中是 `chief_of_staff_agent/CLAUDE.md`
- 把 ClaudeSDKClient 的 `cwd` 参数指向 `CLAUDE.md` 文件所在目录
- 当你希望智能体优先使用高层上下文而非详细数据文件时，用明确的提示词引导它

**重要行为说明**：当 `CLAUDE.md` 和详细数据文件（如 CSV）同时存在时，智能体可能更倾向于读取更细粒度的数据源以给出精确答案。这是预期行为——智能体天然会寻找权威数据。要确保智能体使用 `CLAUDE.md` 的高层上下文，请使用明确的提示词指令（见下方示例）。这带来一个重要启示：`CLAUDE.md` 提供的是*上下文和指导*，而不是对数据源的硬性约束。

In [2]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",  # Points to subdirectory with our CLAUDE.md
        setting_sources=["project"],
    )
) as agent:
    await agent.query("What's our current runway?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# Display the response with HTML rendering
display_agent_response(messages)
# With this prompt, the agent should use CLAUDE.md values: ~$500K burn, 20 months runway

🤖 Thinking...


#### 理解智能体的数据源偏好

**刚才发生了什么：** 通过在提示词中加入，我们引导智能体依赖 `CLAUDE.md` 上下文，而不是从 CSV 文件中寻找更细粒度的数据。

**关键洞察：**
1. **`CLAUDE.md` 是上下文，不是约束**：设置 `cwd` 后，`CLAUDE.md` 文件会作为背景上下文加载。但智能体天然会寻找可获得的最权威数据源。如果存在详细的 CSV 文件，智能体可能为了精确性而优先使用它们。

2. **提示词措辞很重要**：“high-level financial numbers from context”这种说法向智能体表明，你要的是 `CLAUDE.md` 里简化的高管摘要（$500K 资金消耗率、20 个月资金跑道），而不是 financial_data/burn_rate.csv 中逐月的精确数据（$525K 毛消耗、$235K 净消耗）。

3. **架构设计选择**：这种行为在生产系统中其实是理想的——你希望智能体找到最佳数据源。`CLAUDE.md` 应当包含：
   - 高层上下文与策略
   - 公司信息与管理规范
   - 指向详细数据所在位置的指引
   - 关于何时使用高层数字、何时使用详细数字的指引

4. **现实世界的模式**：把 `CLAUDE.md` 看作引导智能体的“入职文档”，而详细文件是智能体在需要精确度时可以查询的“源系统”。

### 特性 1：用于执行 Python 脚本的 Bash 工具

**是什么**：Bash 工具让智能体能够（除其他功能外）直接运行 Python 脚本，从而获得过程性知识、复杂计算、数据分析以及其他超出智能体原生能力的集成。

**为什么**：我们的幕僚长可能需要处理数据文件、运行财务模型，或基于这些数据生成可视化。这些都是使用 Bash 工具的好场景。

**怎么做**：把 Python 脚本放在智能体能访问到的位置，并补充说明它们是什么、如何调用。如果脚本是给幕僚长智能体用的，就把这段说明写进它的 `CLAUDE.md` 文件；如果是给某个子智能体用的，就写进对应的 MD 文件（细节见后文）。在本教程中，我们在 `chief_of_staff_agent/scripts` 中加入了五个示例：
1. `hiring_impact.py`：计算新招聘的工程师对资金消耗率、资金跑道和现金状况的影响。`financial-analyst` 子智能体需要它来针对 $500K 月消耗和 20 个月资金跑道建模招聘场景。
2. `talent_scorer.py`：按加权标准对候选人的技术技能、经验、文化契合度和薪资期望打分。这是 `recruiter` 子智能体的核心工具，用于对照 TechStart 的 $180-220K 高级工程师基准为工程候选人排序。
3. `simple_calculation.py`：对资金跑道、资金消耗率和季度指标做快速财务计算。这是幕僚长获取即时指标的实用脚本，无需复杂建模。
4. `financial_forecast.py`：针对当前 $2.4M 的 ARR 以 15% 月环比增长，建模 ARR 增长场景（基准/乐观/悲观）。这对 `financial-analyst` 预测 B 轮就绪度并验证 $30M 融资目标至关重要。
5. `decision_matrix.py`：为 SmartDev 收购或办公室扩张等战略选择创建加权决策矩阵。帮助幕僚长系统地评估涉及多方干系人和多项标准的复杂决策。

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Bash", "Read"],
        cwd="chief_of_staff_agent",  # Points to subdirectory where our agent is defined
    )
) as agent:
    await agent.query(
        "Use your simple calculation script with a total runway of 2904829 and a monthly burn of 121938."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# Display the response with HTML rendering
display_agent_response(messages)

🤖 Using: Glob()
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Bash()
✓ Tool completed
🤖 Thinking...


Metric,Value
Total Runway,"$2,904,829.00"
Monthly Burn,"$121,938.00"
Runway Months,~23.82 months
Quarterly Burn,"$365,814.00"
Daily Burn Rate,"$4,064.60"


### 特性 2：输出风格

**是什么**：输出风格允许你针对不同受众使用不同的输出风格。每种风格在一个 Markdown 文件中定义。

**为什么**：你的智能体可能被不同专业水平的人使用，他们的关注点也可能不同。输出风格可以区分这些群体，而不必另建一个智能体。

**怎么做**：
- 在 `chief_of_staff_agent/.claude/output-styles/` 中为每种风格配置一个 Markdown 文件。例如查看 `.claude/output-styles/executive.md` 中的 Executive Ouput 风格。输出风格通过简单的前置元数据定义，包含两个字段：name 和 description。注意：确保前置元数据中的 name 与文件名完全一致（区分大小写）

> **重要**：输出风格会修改 Claude Code 底层的系统提示词，省去聚焦软件工程的部分，让你在软件工程工作之外对自己的具体用例有更多控制。

> **SDK 配置说明**：与斜杠命令（见特性 4）类似，输出风格保存在文件系统的 `.claude/output-styles/` 中。要让 SDK 加载这些文件，你**必须**在 `ClaudeAgentOptions` 中包含 `setting_sources=["project"]`。`settings` 参数告诉 SDK *使用哪种*风格，但 `setting_sources` 是真正*加载*风格定义所必需的。这一要求是在调试后续章节时确认的，适用于所有基于文件系统的设置。

In [4]:
messages_executive = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "executive"}',
        # IMPORTANT: setting_sources must include "project" to load output styles from .claude/output-styles/
        # Without this, the SDK does NOT load filesystem settings (output styles, slash commands, etc.)
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_executive.append(msg)

messages_technical = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "technical"}',
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_technical.append(msg)

🤖 Thinking...
🤖 Thinking...


In [5]:
# Display executive style response
display_agent_response(messages_executive)

In [6]:
# Technical output style - detailed, implementation-focused
display_agent_response(messages_technical, title="Technical Style")

### 特性 3：规划模式——只做战略规划而不执行

**是什么**：规划模式要求智能体制定详细的执行计划而不执行任何动作。智能体会分析需求、提出方案、列出步骤，但不修改文件、不执行命令、不做出变更。

**为什么**：复杂任务受益于事先规划，能减少错误、便于评审并改善协调。规划阶段结束后，智能体在执行全程都有一条主线可循。

**怎么做**：只需设置 `permission_mode="plan"`

**计划留存**：由于计划是供评审和决策的有价值产物，我们会演示如何把它们捕获并保存为持久化的 Markdown 文件。这让干系人能在批准执行前审阅计划。

> 说明：这一特性在 Claude Code 中表现出色，但还需要针对使用 SDK 的无界面应用做完整适配。也就是说，智能体会尝试调用它的 `ExitPlanMode()` 工具，而该工具只在交互模式下有意义。在这种情况下，你可以用 `continue_conversation=True` 发送一条后续查询，让智能体在上下文中执行它的计划。

In [7]:
# =============================================================================
# Plan Mode Helper Functions
# =============================================================================
# These utilities handle the various ways an agent might output its plan.
# Since agents can output plans via direct text, Write tool, or Claude's
# internal plan directory, we need robust extraction from multiple sources.

import glob as glob_module
import os
import re
from datetime import datetime
from pathlib import Path
from typing import Any


def extract_plan_from_xml(text: str | None, min_length: int = 200) -> str | None:
    """
    Extract content between <plan> tags from text.

    Args:
        text: The text to search for plan content
        min_length: Minimum character count for valid plan (prevents empty matches)

    Returns:
        Extracted plan content, or None if not found/too short
    """
    if not text:
        return None
    match = re.search(r"<plan>(.*?)</plan>", text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if len(extracted) > min_length:
            return extracted
    return None


def extract_plan_from_messages(
    plan_content: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Try to extract plan from captured message stream content.

    Args:
        plan_content: List of text blocks captured during streaming
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    combined_text = "\n\n".join(plan_content)

    # First try: XML tags
    plan = extract_plan_from_xml(combined_text)
    if plan:
        return plan, "message stream"

    # Fallback: Use raw content if substantial
    if len(combined_text.strip()) > min_fallback_length:
        return combined_text.strip(), "full message content (fallback)"

    return None, None


def extract_plan_from_write_tool(
    write_contents: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Try to extract plan from captured Write tool calls.

    Args:
        write_contents: List of content strings from Write tool calls
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    for content in write_contents:
        # Try XML extraction first
        plan = extract_plan_from_xml(content)
        if plan:
            return plan, "Write tool capture"

        # Fallback: substantial content without tags
        if content and len(content.strip()) > min_fallback_length:
            return content.strip(), "Write tool capture (no XML tags)"

    return None, None


def extract_plan_from_claude_dir(
    max_age_seconds: int = 300, min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Check Claude's internal plan directory for recently created plans.

    Args:
        max_age_seconds: Maximum age of plan file to consider (default: 5 minutes)
        min_fallback_length: Minimum length for fallback (no XML tags)

    Returns:
        Tuple of (plan_text, source_description)
    """
    claude_plans_dir = os.path.expanduser("~/.claude/plans")

    if not os.path.exists(claude_plans_dir):
        return None, None

    # Find most recent plan file
    plan_files = sorted(
        glob_module.glob(os.path.join(claude_plans_dir, "*.md")),
        key=os.path.getmtime,
        reverse=True,
    )

    if not plan_files:
        return None, None

    most_recent = plan_files[0]
    file_age = datetime.now().timestamp() - os.path.getmtime(most_recent)

    if file_age > max_age_seconds:
        return None, None

    with open(most_recent) as f:
        content = f.read()

    filename = os.path.basename(most_recent)

    # Try XML extraction first
    plan = extract_plan_from_xml(content)
    if plan:
        return plan, f"Claude plan file ({filename})"

    # Fallback: substantial content without tags
    if len(content.strip()) > min_fallback_length:
        return content.strip(), f"Claude plan file ({filename}, no XML tags)"

    return None, None


def save_plan_to_file(
    plan_content: str,
    plan_source: str,
    model_name: str,
    prompt_summary: str,
    output_dir: str = "chief_of_staff_agent/plans",
    title: str = "Agent Plan: Engineering Restructure for AI Focus",
) -> Path:
    """
    Save extracted plan to a timestamped markdown file.

    Args:
        plan_content: The plan text to save
        plan_source: Description of where plan was extracted from
        model_name: The model used to generate the plan
        prompt_summary: Brief description of the original prompt
        output_dir: Directory to save plan files
        title: Title for the plan document

    Returns:
        Path to the saved plan file
    """
    plans_dir = Path(output_dir)
    plans_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plan_file = plans_dir / f"plan_{timestamp}.md"

    with open(plan_file, "w") as f:
        f.write(f"# {title}\n\n")
        f.write(f"**Created:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Prompt:** {prompt_summary}\n")
        f.write(f"**Model:** {model_name}\n")
        f.write(f"**Plan Source:** {plan_source}\n\n")
        f.write("---\n\n")
        f.write(plan_content)
        f.write("\n\n---\n\n")
        f.write("*This plan was generated in plan mode and has not been executed.*\n")

    return plan_file


def capture_message_content(
    msg: Any,
    plan_content: list[str],
    write_tool_content: list[str],
    write_tool_paths: list[str],
) -> None:
    """
    Process a streaming message and capture relevant plan content.

    This function extracts content from three potential sources:
    1. Text blocks in message content
    2. Write tool call parameters
    3. Final result attribute

    Args:
        msg: The message object from the agent stream
        plan_content: List to append text content to
        write_tool_content: List to append Write tool content to
        write_tool_paths: List to append Write tool file paths to
    """
    # Source 1: Text blocks from message content
    if hasattr(msg, "content"):
        for block in msg.content:
            if hasattr(block, "text"):
                plan_content.append(block.text)

            # Source 2: Write tool calls
            if hasattr(block, "type") and block.type == "tool_use":
                if hasattr(block, "name") and block.name == "Write":
                    if hasattr(block, "input") and isinstance(block.input, dict):
                        if "content" in block.input:
                            write_tool_content.append(block.input["content"])
                        if "file_path" in block.input:
                            write_tool_paths.append(block.input["file_path"])

    # Source 3: Final result
    if hasattr(msg, "result") and msg.result:
        plan_content.append(msg.result)


print("✅ Plan Mode helper functions loaded")

✅ Plan Mode helper functions loaded


In [8]:
# =============================================================================
# Plan Mode Configuration
# =============================================================================

# Note: MODEL is defined in cell-0 as "claude-opus-4-6"
# Opus excels at complex planning tasks

# The prompt is carefully crafted to:
# 1. Provide explicit context (since Opus prefers explicit information)
# 2. Request XML-tagged output for reliable extraction
# 3. Prevent file-writing so we can capture the plan programmatically

PLAN_PROMPT = """Restructure our engineering team for AI focus.

**CONTEXT (from CLAUDE.md):**
You are the Chief of Staff for TechStart Inc, a 50-person B2B SaaS startup that raised $10M Series A.
- Current engineering team: 25 people (Backend: 12, Frontend: 8, DevOps: 5)
- Monthly burn rate: ~$500K, Runway: 20 months
- Senior Engineer compensation: $180K-$220K + equity

**CRITICAL OUTPUT INSTRUCTIONS:**

1. **DO NOT use the Write tool** - Output your plan directly in your response text
2. **DO NOT save to any files** - I will handle saving the plan myself
3. **Wrap your ENTIRE plan inside `<plan> </plan>` XML tags** in your response

**Required Format:**
<plan>
[Your complete restructuring plan here - include all sections, timelines, budgets, and recommendations]
</plan>

**IMPORTANT:**
- The plan content MUST appear directly in your response between the XML tags
- Do NOT use Write, Edit, or any file-saving tools
- You may research and analyze before outputting, but the final plan must be in your response text
- Include: team structure, hiring recommendations, timeline, budget impact, and success metrics
- Use the company context provided above - do NOT ask clarifying questions"""

print(f"📋 Plan Mode configured with model: {MODEL}")
print(f"📝 Prompt length: {len(PLAN_PROMPT):,} characters")

📋 Plan Mode configured with model: claude-opus-4-6
📝 Prompt length: 1,180 characters


In [9]:
# =============================================================================
# Execute Plan Mode Agent
# =============================================================================
# Run the agent with plan mode enabled. The agent will create a detailed plan
# but won't execute any actions. We capture content from multiple sources
# to handle different agent behaviors.

# Initialize capture lists
messages = []
plan_content = []  # Text from message stream
write_tool_content = []  # Content from Write tool calls
write_tool_paths = []  # Paths from Write tool calls

# Run the agent in plan mode
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        permission_mode="plan",
        cwd="chief_of_staff_agent",
    )
) as agent:
    await agent.query(PLAN_PROMPT)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

        # Capture content from this message
        capture_message_content(msg, plan_content, write_tool_content, write_tool_paths)

print(f"\n✅ Agent completed. Captured {len(plan_content)} content blocks.")

🤖 Thinking...
🤖 Using: ExitPlanMode()
✓ Tool completed
🤖 Thinking...

✅ Agent completed. Captured 3 content blocks.


In [10]:
# =============================================================================
# Extract and Save the Plan
# =============================================================================
# Try multiple sources in priority order to find the plan content.
# This handles different agent behaviors robustly.

final_plan = None
plan_source = None

# Priority 1: Message stream (preferred - direct from agent response)
final_plan, plan_source = extract_plan_from_messages(plan_content)

# Priority 2: Write tool captures (if agent saved despite instructions)
if not final_plan and write_tool_content:
    final_plan, plan_source = extract_plan_from_write_tool(write_tool_content)

# Priority 3: Claude's internal plan directory (safety net)
if not final_plan:
    final_plan, plan_source = extract_plan_from_claude_dir()

# Report results
if final_plan:
    print(f"✅ Plan extracted from: {plan_source}")
    print(f"   Plan length: {len(final_plan):,} characters")

    # Save to file
    plan_file = save_plan_to_file(
        plan_content=final_plan,
        plan_source=plan_source,
        model_name=MODEL,
        prompt_summary="Restructure our engineering team for AI focus.",
    )
    print(f"\n📁 Plan saved to: {plan_file}")
else:
    error_msg = "Could not extract plan content from any source!\n"
    error_msg += "   Sources checked: message stream, Write tool, ~/.claude/plans/"
    if write_tool_paths:
        error_msg += f"\n   Write tool attempted to save to: {write_tool_paths}"
    print(f"❌ ERROR: {error_msg}")
    raise RuntimeError(f"Plan extraction failed: {error_msg}")

✅ Plan extracted from: message stream
   Plan length: 6,783 characters

📁 Plan saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


In [11]:
# Display the plan result with styled HTML
display_agent_response(messages, title="Engineering Restructure Plan")

#### 执行已保存的计划

如上所述，智能体在创建计划后会停止。保存的计划文件作为供干系人评审的产物。

**评审后执行计划的步骤：**
1. 在 `chief_of_staff_agent/plans/plan_*.md` 中审阅已保存的计划
2. 如果获批准，用 `continue_conversation=True` 发送新查询，并移除 `permission_mode="plan"` 以执行

这一工作流形成了“规划 → 评审 → 批准 → 执行”的循环，非常适合组织重组或重大基础设施变更这类高风险决策。

#### 计划留存是如何工作的

在上面的代码中，我们实现了一套**健壮的多来源计划捕获机制**，能处理规划模式智能体输出计划的多种方式：

**挑战：** 使用 `permission_mode="plan"` 时，智能体可能以不同方式输出计划：
1. 在消息流中**直接输出文本**（理想情况）
2. 用 **Write 工具**保存到 `~/.claude/plans/`（Claude 的内部计划系统）
3. 用 **Write 工具**保存到自定义路径

我们的捕获机制通过**基于优先级的回退体系**处理上述三种场景：

**来源优先级（按顺序）：**

1. **消息流**（首选）
   - 在流式输出期间从 `msg.content` 捕获文本块
   - 提取 `<plan></plan>` XML 标签之间的内容
   - 这是最干净的做法，因为内容直接来自响应

2. **Write 工具捕获**
   - 在消息流中监视 Write 工具调用
   - 提取正在写入的 `content` 参数
   - 当智能体不顾提示词指令仍决定保存时，这一途径很有用

3. **Claude 的内部计划目录**
   - 检查 `~/.claude/plans/` 中最近创建的计划文件（5 分钟以内）
   - 从最新的文件读取并提取内容
   - 在其他方法失败时充当安全网

4. **完整内容回退**
   - 如果未找到 XML 标签但存在大量内容（超过 500 字符），直接使用它
   - 避免生成空的计划文件，同时保留部分信息

**关键实现细节：**

```python
def extract_plan_from_text(text):
    """Extract content between <plan> tags, return None if not found or empty."""
    match = re.search(r'<plan>(.*?)</plan>', text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        # Validate minimum content length (a real plan should be substantial)
        if len(extracted) > 200:
            return extracted
    return None
```

**为什么内容校验很重要：**
- 早期版本可能在提取“成功”但无内容时生成空的计划文件
- 现在我们要求带 XML 标签的内容至少 200 个字符
- 这避免了正则匹配到空内容或无意义内容造成的误判

**面向直接输出的提示词工程：** 提示词明确要求智能体：
- **不要使用 Write 工具**——避免绕道文件系统
- **直接在响应中输出**——确保内容通过消息流传递
- **使用 XML 标签**——便于从可能冗长的响应中干净地提取内容

这种做法给你带来：
- **可靠性**：无论智能体行为如何，计划都能被捕获
- **透明度**：保存的文件会标明使用了哪个来源
- **审计线索**：所有计划的历史记录，含时间戳和来源元数据
- **调试能力**：提取失败时给出清晰的错误信息

下面查看已保存的计划：

In [15]:
# Display the saved plan with markdown rendering
from IPython.display import Markdown, display

# Show the plan with proper markdown formatting
display(Markdown(f"## 📋 Saved Plan Preview\n\n{final_plan}"))

print(f"\n📁 Full plan with metadata saved to: {plan_file}")

## 📋 Saved Plan Preview

# TechStart Inc. Engineering Team Restructuring Plan for AI Focus

## Executive Summary
Restructure the 25-person engineering team to build AI/ML capabilities while maintaining core product development. This plan balances immediate hiring needs with internal upskilling and strategic reorganization over a 12-month period.

---

## Current State Assessment

| Team | Headcount | Current Focus |
|------|-----------|---------------|
| Backend | 12 | Core product, APIs, infrastructure |
| Frontend | 8 | Web/mobile interfaces |
| DevOps | 5 | CI/CD, cloud infrastructure |
| **Total** | **25** | |

**Key Constraints:**
- Monthly burn: ~$500K | Runway: 20 months
- Senior Engineer comp: $180K-$220K + equity
- Series A stage - need to show growth metrics

---

## Proposed Future State (Month 12)

### New Team Structure

| Team | Current | Future | Change |
|------|---------|--------|--------|
| Backend/Core | 12 | 8 | -4 |
| Frontend | 8 | 6 | -2 |
| DevOps/MLOps | 5 | 6 | +1 |
| **AI/ML Platform** | 0 | 5 | +5 |
| **AI Product** | 0 | 4 | +4 |
| **Total** | **25** | **29** | **+4** |

### New Teams Created

**1. AI/ML Platform Team (5 engineers)**
- Focus: Infrastructure, model training pipelines, ML tooling
- Composition: 2 new hires (ML Engineers) + 2 internal transfers (Backend) + 1 internal (DevOps)
- Lead: New hire - Senior ML Engineer ($200-220K)

**2. AI Product Team (4 engineers)**
- Focus: AI features, integrations, user-facing ML applications
- Composition: 2 new hires (ML/AI specialists) + 1 internal (Backend) + 1 internal (Frontend)
- Lead: Promoted internal senior engineer + AI upskilling

---

## Hiring Plan

### New Positions (6 total new hires)

| Role | Priority | Timeline | Comp Range | Notes |
|------|----------|----------|------------|-------|
| Senior ML Engineer (Lead) | P0 | Month 1-2 | $200-220K | Team lead, architecture |
| ML Engineer | P0 | Month 2-3 | $180-200K | Platform focus |
| ML Engineer | P1 | Month 3-4 | $180-200K | Product focus |
| AI/ML Engineer | P1 | Month 4-5 | $170-190K | Generalist |
| MLOps Engineer | P2 | Month 5-6 | $160-180K | DevOps + ML |
| Junior ML Engineer | P2 | Month 6-8 | $130-150K | Growth hire |

**Total New Hiring Cost:** ~$1.02-1.14M annually

### Internal Transfers & Upskilling (4 engineers)

| From Team | # Engineers | New Role | Training Investment |
|-----------|-------------|----------|---------------------|
| Backend | 3 | ML Platform/Product | $15K each (courses, certs) |
| Frontend | 1 | AI Product (UI/UX) | $10K (AI tools training) |

**Upskilling Budget:** ~$55K total

---

## Implementation Timeline

### Phase 1: Foundation (Months 1-3)
- [ ] Hire Senior ML Engineer (Team Lead) - **Critical first hire**
- [ ] Identify 4 internal transfer candidates based on interest/aptitude
- [ ] Begin ML upskilling program for transfer candidates
- [ ] Set up ML infrastructure foundations (MLOps engineer involvement)
- [ ] Define AI product roadmap with Product team

### Phase 2: Team Formation (Months 4-6)
- [ ] Complete core AI team hiring (4 of 6 hires)
- [ ] Officially launch AI/ML Platform team
- [ ] Internal transfers complete bootcamp/training
- [ ] First AI feature POC delivered
- [ ] MLOps practices integrated into DevOps workflow

### Phase 3: Scaling (Months 7-9)
- [ ] Complete remaining hires
- [ ] AI Product team fully operational
- [ ] First AI feature in production
- [ ] Cross-team collaboration patterns established
- [ ] Model monitoring and observability in place

### Phase 4: Optimization (Months 10-12)
- [ ] Team velocity optimization
- [ ] Evaluate team composition effectiveness
- [ ] Plan for next growth phase
- [ ] Document AI development best practices

---

## Budget Impact Analysis

### Monthly Cost Changes

| Category | Current | Month 6 | Month 12 |
|----------|---------|---------|----------|
| Engineering Salaries | ~$375K | ~$430K | ~$485K |
| Training/Upskilling | $0 | $9K | $2K |
| AI Tools/Infrastructure | ~$5K | ~$20K | ~$35K |
| Recruiting Costs | Variable | ~$40K | ~$15K |
| **Monthly Total** | **~$380K** | **~$499K** | **~$537K** |

### Annual Impact Summary
- **Year 1 Additional Investment:** ~$900K-1.1M
- **New Monthly Burn (Month 12):** ~$537K (+$37K from engineering growth)
- **Runway Impact:** Reduces to ~17 months (still healthy for Series A)

### ROI Considerations
- AI features typically command 20-40% pricing premium in B2B SaaS
- Competitive differentiation in market
- Potential for AI-driven operational efficiencies

---

## Risk Mitigation

| Risk | Likelihood | Impact | Mitigation |
|------|------------|--------|------------|
| Senior ML hire takes >3 months | Medium | High | Engage specialized recruiters early; consider contractor bridge |
| Internal transfers struggle with ML | Low | Medium | Rigorous selection process; extended training runway |
| AI projects don't deliver value | Medium | High | Start with high-impact, lower-complexity features |
| Team culture friction | Low | Medium | Integrate teams gradually; shared goals and rituals |
| Budget overrun | Medium | Medium | Phase hiring based on runway checks quarterly |

---

## Success Metrics

### 6-Month Milestones
- [ ] AI/ML Platform team fully staffed (5 engineers)
- [ ] First AI-powered feature in beta
- [ ] 4 internal engineers completed ML certification
- [ ] ML infrastructure supporting model training/deployment

### 12-Month Milestones
- [ ] 2+ AI features in production
- [ ] AI team velocity matches core team benchmarks
- [ ] Customer NPS improvement attributable to AI features
- [ ] 15%+ of product roadmap is AI-focused
- [ ] Retention of transferred engineers >90%

### KPIs to Track
- Time-to-hire for ML roles
- AI feature adoption rates
- Model performance metrics (latency, accuracy)
- Engineering velocity (story points/sprint) across teams
- Employee satisfaction scores (especially transfers)

---

## Immediate Next Steps (Week 1-2)

1. **Executive alignment** - Present plan to leadership for approval
2. **Recruiter engagement** - Brief recruiting on Senior ML Engineer search
3. **Internal survey** - Gauge interest in AI/ML roles among current engineers
4. **Budget approval** - Finance sign-off on increased burn rate
5. **Infrastructure assessment** - DevOps audit of ML infrastructure needs

---

## Organizational Chart (Future State)

```
VP Engineering
├── Backend/Core Team (8)
│   └── 2 squads × 4 engineers
├── Frontend Team (6)
│   └── 2 squads × 3 engineers
├── DevOps/MLOps Team (6)
│   └── 4 DevOps + 2 MLOps
├── AI/ML Platform Team (5) [NEW]
│   └── Lead + 4 engineers
└── AI Product Team (4) [NEW]
    └── Lead + 3 engineers
```

---

*Plan prepared for TechStart Inc. Series A stage (~$10M raised, 20-month runway)*
*Recommended review: Quarterly budget and hiring progress checkpoints*


📁 Full plan with metadata saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


## 高级特性

### 特性 4：自定义斜杠命令

> 说明：斜杠命令是面向用户的语法糖，不是新的智能体能力

**是什么**：自定义斜杠命令是预定义的提示词模板，用户可以用简写语法触发（例如 `/budget-impact`）。这些是**面向用户的快捷方式**，不是智能体能力。可以把它们理解为会展开成完整、精心撰写的提示词的快捷键。

**为什么**：你的幕僚长会处理反复出现的高管问题。用户不必反复输入复杂提示词，而可以使用已经审核过的提示词。这提升了一致性和标准化。

**怎么做**：
- 在 `.claude/commands/` 中定义一个 Markdown 文件。例如我们在 `.claude/commands/slash-command-test.md` 中定义了一个。注意命令的定义方式：包含两个字段（name、description）的前置元数据，以及展开后的提示词，并可选择把查询中传入的参数包含进来。
- 可以用 `$ARGUMENTS`（完整参数串）或 `$1`、`$2` 等（位置参数）为提示词添加参数
- 用户在提示词中使用该斜杠命令

> **关键 SDK 配置**：使用 SDK 时，你**必须**在 `ClaudeAgentOptions` 中设置 `setting_sources=["project"]`，斜杠命令才能生效。默认情况下，SDK 以隔离模式运行，不会加载文件系统设置（斜杠命令、CLAUDE.md、子智能体、钩子等）。这与交互式使用 Claude Code 时自动加载这些设置的做法不同。

In [16]:
# User types: "/slash-command-test this is a test"
# -> behind the scenes EXPANDS to the prompt in .claude/commands/slash-command-test.md
# In this case the expanded prompt says to simply reverse the sentence word wise

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        # IMPORTANT: setting_sources must include "project" to load slash commands from .claude/commands/
        # Without this, the SDK does NOT load filesystem settings (slash commands, CLAUDE.md, etc.)
        setting_sources=["project"],
    )
) as agent:
    await agent.query("/slash-command-test this is a test")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...


In [17]:
display_agent_response(messages, title="Slash Command Result")

### 特性 5：钩子——自动化的确定性动作

**是什么**：钩子是 Python 脚本，可以设置为在若干事件（尤其是特定工具调用之前（pre）或之后（post））自动执行。钩子**确定性地**运行，非常适合校验和审计线索。

**为什么**：设想你需要确保智能体具备某些护栏（例如阻止危险操作），或者需要一条审计线索的场景。钩子与智能体配合使用最为理想：既给智能体足够的自由度完成任务，又确保其行为安全。

**怎么做**：
- 在 `.claude/hooks/` 中定义钩子脚本 -> 钩子被触发时应执行_什么_行为
- 在 `.claude/settings.local.json` 中定义钩子配置 -> 钩子应在_何时_被触发
- 在本例中，我们的钩子被配置为监视特定工具调用（Bash、Write、Edit）
- 当这些工具被调用时，钩子脚本在工具完成后运行（PostToolUse）

> **SDK 配置说明**：在 `.claude/settings.local.json` 中配置的钩子需要 `setting_sources=["project", "local"]`。SDK 区分三种设置来源：
> - `"project"` → `.claude/settings.json`（纳入版本控制、团队共享）
> - `"local"` → `.claude/settings.local.json`（被 git 忽略、本地设置，如钩子）
> - `"user"` → `~/.claude/settings.json`（全局用户设置）
>
> 由于我们的钩子在 `settings.local.json` 中，必须在 `setting_sources` 中包含 `"local"`。

**示例：用于合规的报告跟踪**

一个钩子，用于记录对财务报告执行的 Write/Edit 操作，以便审计与合规。该钩子定义在 `chief_of_staff_agent/.claude/hooks/report-tracker.py` 中，而强制执行它的逻辑位于 `chief_of_staff_agent/.claude/settings.local.json`：


```json
"hooks": {
  "PostToolUse": [
    {
      "matcher": "Write",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    },
    {
      "matcher": "Edit",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    }
  ]
}
```

In [18]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        allowed_tools=["Bash", "Write", "Edit", "MultiEdit"],
        # IMPORTANT: setting_sources must include BOTH "project" AND "local" to load hooks
        # - "project" loads .claude/settings.json (shared settings, CLAUDE.md, slash commands)
        # - "local" loads .claude/settings.local.json (where hooks are configured)
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query(
        "Create a quick Q2 financial forecast report with our current burn rate and runway projections. Save it to our /output_reports folder."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# The hook will track this in audit/report_history.json
display_agent_response(messages, title="Q2 Financial Forecast")

🤖 Using: Bash()
🤖 Using: Bash()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


Metric,Value
Cash in Bank,$10M
Current Runway,20 months (until Sept 2025)
Monthly Burn,$500K gross / ~$260K net
Q2 Total Net Burn,$740K
ARR,$2.4M (15% MoM growth)


现在打开 `./chief_of_staff_agent/audit/report_history.json`，你会发现它已记录智能体创建或修改了你的报告。生成的报告本身位于 `./chief_of_staff_agent/output_reports/`。

### 特性 6：通过 Task 工具使用子智能体

**是什么**：Task 工具让你的智能体可以把专业化工作分派给其他子智能体。这些子智能体各自拥有自己的指令、工具和专业领域。

**为什么**：加入子智能体打开了很多可能性：
1. 专业化：每个子智能体都是其领域的专家
2. 独立上下文：子智能体拥有自己的对话历史和工具
3. 并行化：多个子智能体可以同时处理不同方面。

**怎么做**：
- 把 `"Task"` 加入 allowed_tools
- 用系统提示词指导你的智能体如何分派任务（也可以更笼统地写在它的 CLAUDE.md 中）
- 在 `.claude/agents/` 中为每个智能体创建一个 Markdown 文件。例如查看 `.claude/agents/financial-analyst.md`，注意一个（子）智能体可以如何用如此简单直观的 Markdown 文件定义：包含三个字段（name、description、tools）的前置元数据，以及它的系统提示词。description 有助于主幕僚长智能体知道何时调用每个子智能体。

**可视化增强**：我们的 `print_activity()` 和 `visualize_conversation()` 工具已增强，能清晰展示子智能体的操作：
- 🚀 表示正在分派给某个子智能体（并标出子智能体名称）
- 📎 表示由子智能体使用的工具（缩进以体现视觉层次）
- 视觉分隔线清晰标出子智能体分派与完成的边界
- 任务描述和提示词会显示在对话时间线中

In [19]:
# Reset the subagent tracking context before starting a new query
# This ensures clean state for activity display
reset_activity_context()

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Task"],  # this enables our Chief agent to invoke subagents
        system_prompt="Delegate financial questions to the financial-analyst subagent. Do not try to answer these questions yourself.",
        cwd="chief_of_staff_agent",
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query("Should we hire 5 engineers? Analyze the financial impact.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages, title="Hiring Impact Analysis")

🤖 Thinking...
🚀 Delegating to subagent: financial-analyst
   └─ Task: Analyze 5 engineer hiring impact
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Read()
   ✓ Tool completed
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Thinking...


Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


In [20]:
visualize_conversation(messages)

Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


在这里，当主智能体决定使用某个子智能体时，它会：
  1. 调用 Task 工具，并传入如下参数：
  ```json
    {
      "description": "Analyze hiring impact",
      "prompt": "Analyze the financial impact of hiring 5 engineers...",
      "subagent_type": "financial-analyst"
    }
  ```
  2. Task 工具在独立上下文中执行该子智能体
  3. 将结果返回给主幕僚长智能体以继续处理

## 把所有特性组合起来

现在把前面看到的一切组合起来。我们会让智能体判断招聘 3 名高级工程师的财务影响，并把洞察写入 `output_reports/hiring_decision.md`。这一示例展示了上述全部特性：
- **Bash 工具**：用于执行 `hiring_impact.py` 脚本，判断招聘新工程师的影响
- **记忆**：读取目录中的 `CLAUDE.md` 作为上下文，理解当前的预算、资金跑道、收入等相关信息
- **输出风格**：在 `chief_of_staff_agent/.claude/output-styles` 中定义的不同输出风格
- **自定义斜杠命令**：使用快捷方式 `/budget-impact`，它展开为 `chief_of_staff_agent/.claude/commands` 中定义的完整提示词
- **子智能体**：我们的 `/budget_impact` 命令引导幕僚长智能体调用在 `chief_of_staff_agent/.claude/agents` 中定义的 financial-analyst 子智能体
- **钩子**：钩子定义在 `chief_of_staff_agent/.claude/hooks` 中，并在 `chief_of_staff_agent/.claude/settings.local.json` 中配置
    - 如果我们的某个智能体正在更新财务报告，钩子应在 `chief_of_staff_agent/audit/report_history.json` 日志文件中记录这次编辑/写入活动
    - 如果财务分析师子智能体调用 `hiring_impact.py` 脚本，这会被记录在 `chief_of_staff_agent/audit/tool_usage_log.json` 日志文件中

- **规划模式**：如果你希望幕僚长先提出一个计划供你批准再采取任何行动，请取消下面被注释掉的那一行

为了开箱即用，我们把智能体循环封装在一个 Python 文件中，与上一个笔记本中的做法类似。请查看 `chief_of_staff_agent` 子目录中的 agent.py 文件。

总体而言，我们的 `send_query()` 函数接收 4 个参数（prompt、continue_conversation、permission_mode 和 output_style），其余内容都在智能体文件中设置，即：系统提示词、最大轮数、允许的工具和工作目录。

为了更好地直观理解这一切如何协同，请查看这些 [Claude 为我们绘制的流程与架构图 :)](./chief_of_staff_agent/flow_diagram.md)


In [21]:
from chief_of_staff_agent.agent import send_query

reset_activity_context()

result, messages = await send_query(
    "/budget-impact hiring 3 senior engineers. Save your insights by updating the 'hiring_decision.md' file in /output_reports or creating a new file there",
    # permission_mode="plan", # Enable this to use planning mode
    output_style="executive",
)

🤖 Thinking...
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Task()
✓ Tool completed
🤖 Using: Bash()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


In [22]:
visualize_conversation(messages)

Metric,Current,Post-Hiring,Change
Monthly Gross Burn,$525K,$590K,+$65K (+12.4%)
Monthly Net Burn,$235K,$300K,+$65K (+27.7%)
Cash Runway,42.6 months,32.9 months,-9.7 months
Break-Even,-,November 2024,5 months


## 结论

我们演示了 Claude Code SDK 如何让你构建具备企业级特性的复杂多智能体系统。从使用 Bash 工具执行脚本这一基础能力出发，我们逐步引入了高级能力，包括用 CLAUDE.md 实现持久记忆、面向不同受众的自定义输出风格、战略规划模式、便于用户使用的斜杠命令、用于护栏的合规钩子，以及用于专业化任务的子智能体协调。

通过组合这些特性，我们创建了一个能够处理复杂高管决策工作流的 AI 幕僚长。该系统把财务分析分派给专业化的子智能体，通过钩子维护审计线索，为不同干系人调整沟通风格，并提供有数据驱动分析支撑的可执行洞察。

这些高级智能体模式与多智能体编排的基础，为你构建可投产的企业级系统做好准备。在下一个笔记本中，我们将探索如何通过 Model Context Protocol（MCP）服务器把智能体连接到外部服务，从而大幅扩展其超出内置工具的能力。

下一篇：[02_The_observability_agent.ipynb](02_The_observability_agent.ipynb) - 学习如何通过 MCP 用自定义集成和外部数据源扩展你的智能体。